# Zip code extraction timings
Run this notebook on Onyxia after the bronze users file has been uploaded.


In [ ]:
import os
import re
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

bucket = os.environ['KUBERNETES_NAMESPACE']
spark = (SparkSession.builder.appName('zip-timeit').master('local[2]')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.hadoop.fs.s3a.endpoint', f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
    .config('spark.hadoop.fs.s3a.session.token', os.environ['AWS_SESSION_TOKEN'])
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
users_bronze = spark.read.csv(f's3a://{bucket}/bronze/users.csv',
                              header=True, inferSchema=True,
                              samplingRatio=0.01, multiLine=True)

@F.udf(StringType())
def zip_python(address):
    match = re.search(r'([A-Z]{2}) ([0-9]{5})$', address or '')
    return match.group(2) if match else ''

@F.pandas_udf(StringType())
def zip_pandas(address: pd.Series) -> pd.Series:
    return address.str.extract(r'([A-Z]{2}) ([0-9]{5})$')[1].fillna('')


In [ ]:
%%timeit -n 1 -r 3
zip_codes = users_bronze.select(F.regexp_extract('address', r'([A-Z]{2}) ([0-9]{5})$', 2).alias('zip_code'))
zip_codes.filter(F.col('zip_code') != '').count()


In [ ]:
%%timeit -n 1 -r 3
zip_codes = users_bronze.select(zip_python('address').alias('zip_code'))
zip_codes.filter(F.col('zip_code') != '').count()


In [ ]:
%%timeit -n 1 -r 3
zip_codes = users_bronze.select(zip_pandas('address').alias('zip_code'))
zip_codes.filter(F.col('zip_code') != '').count()


In [ ]:
spark.stop()
